In [1]:
import pandas as pd
import h3
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Načtení dat
df = pd.read_csv('data/chains.csv')
print(f"Načteno {len(df)} řádků")
print(f"Řetězce: {df['q'].unique()}")
df.head()


Načteno 1239 řádků
Řetězce: ['costco' 'amc' 'sams']


,q,lat,lng,region_res5,region_res4
0,costco,47.355381,-122.121914,8528d437fffffff,8428d43ffffffff
1,costco,39.914269,-86.061598,852a924ffffffff,842a925ffffffff
2,costco,40.004032,-86.000632,852a924bfffffff,842a925ffffffff
3,costco,42.432936,-83.429501,852ab2cbfffffff,842ab2dffffffff
4,costco,34.427788,-119.874785,8529ac93fffffff,8429ac9ffffffff


In [2]:
# Funkce pro získání polygonu H3 hexagonu
def get_h3_polygon(h3_hex):
    """Převede H3 hexagon na polygon pro zobrazení na mapě"""
    try:
        # Získání hranic hexagonu - vrací tuple (lat, lng)
        boundary = h3.cell_to_boundary(h3_hex)
        # Plotly očekává [lng, lat] formát
        return [[lng, lat] for lat, lng in boundary]
    except Exception as e:
        print(f"Chyba při získávání polygonu pro {h3_hex}: {e}")
        return None

# Funkce pro získání všech polygonů pro daný řetězec a resolution
def get_polygons_for_chain(df, chain_name, resolution_col):
    """Vrátí seznam polygonů pro daný řetězec a resolution"""
    chain_data = df[df['q'] == chain_name]
    polygons = []
    for h3_hex in chain_data[resolution_col].unique():
        if pd.notna(h3_hex):
            poly = get_h3_polygon(h3_hex)
            if poly:
                polygons.append(poly)
    return polygons

# Barvy pro jednotlivé řetězce
chain_colors = {
    'costco': '#FF6B6B',      # Červená
    'amc': '#4ECDC4',         # Tyrkysová
    'sams': '#45B7D1'         # Modrá
}

print("Funkce připraveny")


Funkce připraveny


In [3]:
# Vytvoření mapy
fig = go.Figure()

# Pro každý řetězec
for chain_name in df['q'].unique():
    color = chain_colors.get(chain_name, '#808080')
    
    # R4 polygony (světlejší, s průhledností) - nejdřív, aby byly pod R5
    r4_polygons = get_polygons_for_chain(df, chain_name, 'region_res4')
    for i, poly in enumerate(r4_polygons):
        fig.add_trace(go.Scattermapbox(
            mode='lines',
            lon=[p[0] for p in poly] + [poly[0][0]],  # Uzavření polygonu
            lat=[p[1] for p in poly] + [poly[0][1]],
            fill='toself',
            fillcolor=color,
            line=dict(color=color, width=1),
            opacity=0.3,  # Průhlednost pro R4
            name=f'{chain_name} (R4)' if i == 0 else '',  # Legenda jen pro první
            showlegend=(i == 0),  # Zobrazit legendu jen pro první polygon
            legendgroup=f'{chain_name}_r4',
            hoverinfo='skip'
        ))
    
    # R5 polygony (tmavší, méně průhledné) - nahoře
    r5_polygons = get_polygons_for_chain(df, chain_name, 'region_res5')
    for i, poly in enumerate(r5_polygons):
        fig.add_trace(go.Scattermapbox(
            mode='lines',
            lon=[p[0] for p in poly] + [poly[0][0]],  # Uzavření polygonu
            lat=[p[1] for p in poly] + [poly[0][1]],
            fill='toself',
            fillcolor=color,
            line=dict(color=color, width=2),
            opacity=0.7,  # Méně průhledné pro R5
            name=f'{chain_name} (R5)' if i == 0 else '',  # Legenda jen pro první
            showlegend=(i == 0),  # Zobrazit legendu jen pro první polygon
            legendgroup=f'{chain_name}_r5',
            hoverinfo='skip'
        ))

# Nastavení mapy
fig.update_layout(
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=39.8283, lon=-98.5795),  # Střed USA
        zoom=4
    ),
    title='Pobočky řetězců - H3 polygony (R4 světlejší, R5 tmavší)',
    height=800,
    showlegend=True
)

fig.show()
